# Frontend Architecture & Backend Integration Guide

This document details the frontend implementation, focusing on authentication, validation logic, and API contracts. It serves as a technical reference for backend developers to ensure seamless integration.

---

## 1. Authentication & Security (JWT)

The frontend uses a **JWT (JSON Web Token)** based authentication system.

### Token Management

Tokens are persisted in `localStorage` to maintain sessions across page reloads.

| Item              | Key              | Description                                      |
| ----------------- | ---------------- | ------------------------------------------------ |
| **Access Token**  | `auth_token`     | Short-lived JWT attached to API requests.         |
| **Refresh Token** | `refresh_token`  | Long-lived JWT used to refresh the session.       |
| **User Profile**  | `customer_user`  | JSON object containing `id`, `email`, and `role`. |

### Axios Interceptor Implementation

All HTTP requests are handled via a centralized Axios instance (`src/services/api.js`). Interceptors automatically inject the token and handle 401 errors.

```javascript
// Request Interceptor: Attaches Token
api.interceptors.request.use((config) => {
  const token = localStorage.getItem('auth_token');
  if (token) config.headers.Authorization = `Bearer ${token}`;
  return config;
});

// Response Interceptor: Handles 401 Unauthorized
api.interceptors.response.use(
  (response) => response,
  (error) => {
    if (error.response?.status === 401) {
      localStorage.removeItem('auth_token');
      localStorage.removeItem('refresh_token');
      localStorage.removeItem('customer_user');
      window.location.href = '/login';
    }
    return Promise.reject(error);
  }
);
```

### Auth Bypass (Dev Tool)

For testing without a backend, the frontend supports query parameters:

- `?bypassAuth=owner`
- `?bypassAuth=warehouse`
- `?bypassAuth=admin`

---

## 2. Form Validation Rules

The frontend enforces strict validation **before** sending data to the API. The backend should implement redundant validation.

### Registration Form (inside `/login` toggle)

| Field                | Validation Logic / Regex                                        | Error Message                                                                       |
| -------------------- | --------------------------------------------------------------- | ----------------------------------------------------------------------------------- |
| **Email**            | Must end with `@gmail.com` or `.edu.np`                         | "Email must end with @gmail.com or .edu.np"                                         |
| **Password**         | Min 8 chars, 1 Upper, 1 Lower, 1 Number, 1 Special             | "Password must be 8+ chars with uppercase, lowercase, number, and special character"|
| **Confirm Password** | `value === password`                                            | "Password and confirm password do not match"                                        |
| **Phone**            | `/^\d{10}$/`                                                    | "Phone number must be exactly 10 digits"                                            |
| **Date of Birth**    | `age >= 16`                                                     | "You must be at least 16 years old to sign up"                                      |
| **Required Fields**  | First Name, Last Name, Gender, Address                          | "This field is required" / "Please select your gender"                              |

### Login Form (default view on `/login`)

- **Email**: Required.
- **Password**: Required.

> **Note:** Registration and Login share the same page (`/login`) with a toggle. There is no separate `/register` route — this is by design.

---

## 3. API Contracts & Payloads

### Login Endpoint

**URL**: `POST /api/auth/login/`

```json
{ "email": "user@example.com", "password": "SecretPassword123!" }
```

### Registration Endpoint

**URL**: `POST /api/auth/register/`
**Note:** Frontend sends **camelCase** keys.

```json
{
  "firstName": "John",
  "lastName": "Doe",
  "email": "john@gmail.com",
  "password": "Secret@123",
  "confirmPassword": "Secret@123",
  "phone": "9812345678",
  "dob": "2000-01-01",
  "gender": "male",
  "address": "Kathmandu",
  "role": "customer"
}
```

### Expected Auth Response

The backend **must** return the user object with the `role` field.

```json
{
  "access": "eyJhbGci...",
  "refresh": "eyJhbGci...",
  "user": {
    "id": 1,
    "email": "john@gmail.com",
    "role": "owner",
    "firstName": "John",
    "lastName": "Doe"
  }
}
```

Supported roles: `customer`, `owner`, `warehouse`, `admin`

---

## 4. Role-Based Routing

Each role section is wrapped in a **guarded layout component** that checks `user.role` from `AuthContext`. Unauthorized users are redirected to `/login`.

### Customer Routes (public)

| Route            | Page             | Description                   |
| ---------------- | ---------------- | ----------------------------- |
| `/`              | Home             | Hero, categories, products    |
| `/product/:id`   | ProductDetail    | Full product page with specs  |
| `/wishlist`      | Wishlist         | Saved products                |
| `/cart`          | Cart             | Cart management               |
| `/compare`       | Compare          | Side-by-side comparison       |
| `/checkout`      | Checkout         | Shipping + payment flow       |
| `/login`         | Login            | Login / Signup toggle         |
| `/profile`       | Profile          | User info + logout            |

### Owner Routes (`/owner/*` — requires `role === 'owner'`)

| Route               | Page               |
| -------------------- | -------------------- |
| `/owner/dashboard`   | OwnerDashboard       |
| `/owner/products`    | ProductManagement    |
| `/owner/orders`      | OrderManagement      |
| `/owner/analytics`   | Analytics (Plotly)   |

### Warehouse Routes (`/warehouse/*` — requires `role === 'warehouse'`)

| Route                         | Page                 |
| ----------------------------- | -------------------- |
| `/warehouse/dashboard`        | WarehouseDashboard   |
| `/warehouse/inventory`        | InventoryManagement  |
| `/warehouse/stock-movements`  | StockMovements       |
| `/warehouse/low-stock-alerts` | LowStockAlerts       |

### Admin Routes (`/admin/*` — requires `role === 'admin'`)

| Route               | Page               |
| -------------------- | -------------------- |
| `/admin/dashboard`   | AdminDashboard       |
| `/admin/users`       | UserManagement       |
| `/admin/suppliers`   | SupplierManagement   |
| `/admin/logs`        | SystemLogs           |
| `/admin/analytics`   | AnalyticsSummary     |

---

## 5. Tech Stack

| Concern         | Library / Tool                              |
| --------------- | ------------------------------------------- |
| Framework       | React 19 + Vite 7                           |
| Routing         | React Router DOM 7                          |
| HTTP Client     | Axios (interceptors for JWT)                |
| State           | React Context API (`AuthContext`)            |
| Charts          | Plotly (`react-plotly.js`) + Recharts        |
| Icons           | Lucide React, React Icons                   |
| Date Utils      | date-fns                                    |
| Styling         | Tailwind CSS 4 + component-level CSS-in-JS  |

---

## 6. API Service Structure

All API calls are centralized in `src/services/api.js` and grouped by role:

- **`authAPI`** — login, register, logout, refresh
- **`customerAPI`** — products, cart, wishlist, orders, profile, reviews
- **`ownerAPI`** — dashboard analytics, product CRUD, order management
- **`warehouseAPI`** — inventory, stock movements, alerts, suppliers
- **`adminAPI`** — users, suppliers, logs, analytics

> **Note:** Customer-facing pages (Home, Cart, Wishlist, Compare, Checkout) currently use **local React state** for cart/wishlist/compare data. The `customerAPI` endpoints are fully defined and ready — they will be connected once the backend is live.
